In [1]:
# ============================================================
# 08_trade_candidate_mapping_v0_1
#
# Goal:
#   Convert daily put/call trade signals into naked option
#   trade candidates.
#
# Stage 1:
#   Load and validate signal files.
#
# Stage 2:
#   Build raw candidate rows with target expiry dates.
#
# No option pricing yet.
# No P&L yet.
# No risk sizing yet.
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RAW_DATA_DIR = DATA_DIR / "raw"
AUDIT_DIR = DATA_DIR / "audit"

DAILY_SIGNAL_SUMMARY_PARQUET = PROCESSED_DATA_DIR / "daily_trade_signal_summary_v0_1.parquet"
TRADE_SIGNAL_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_signal_panel_v0_1.parquet"

TRADE_CANDIDATE_RAW_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1_raw_targets.csv"
TRADE_CANDIDATE_RAW_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1_raw_targets.parquet"

print("Project root:", PROJECT_ROOT)
print("Daily signal summary exists:", DAILY_SIGNAL_SUMMARY_PARQUET.exists())
print("Trade signal panel exists:", TRADE_SIGNAL_PANEL_PARQUET.exists())
print("Raw data dir exists:", RAW_DATA_DIR.exists())

Project root: C:\Users\patri\vrp_project
Daily signal summary exists: True
Trade signal panel exists: True
Raw data dir exists: True


In [2]:
# ============================================================
# Load daily signal summary from notebook 07
# ============================================================

daily_signal_df = pd.read_parquet(DAILY_SIGNAL_SUMMARY_PARQUET).copy()

required_daily_cols = [
    "trade_date",
    "spx_close",
    "spx_rsi_14",
    "signal_state",

    "selected_put_signal",
    "selected_put_tenor",
    "selected_put_tier",
    "selected_put_primary_vrp_signal",
    "selected_put_blended_z",
    "selected_put_3m_z",
    "selected_put_1y_z",
    "selected_put_implied_vol",
    "selected_put_trailing_rv",
    "selected_put_variance_ratio",
    "selected_put_vol_ratio",

    "selected_call_signal",
    "selected_call_tenor",
    "selected_call_primary_vrp_signal",
    "selected_call_blended_z",
    "selected_call_3m_z",
    "selected_call_1y_z",
    "selected_call_implied_vol",
    "selected_call_trailing_rv",
    "selected_call_variance_ratio",
    "selected_call_vol_ratio",

    "preferred_tenor",
]

missing_daily_cols = [c for c in required_daily_cols if c not in daily_signal_df.columns]

if missing_daily_cols:
    raise ValueError(f"Missing required daily columns: {missing_daily_cols}")

print("Daily signal rows:", len(daily_signal_df))
print("Date range:", daily_signal_df["trade_date"].min(), "to", daily_signal_df["trade_date"].max())

print("\nSignal state counts:")
display(daily_signal_df["signal_state"].value_counts().rename("count").to_frame())

print("\nSelected put signal dates:", daily_signal_df["selected_put_signal"].sum())
print("Selected call signal dates:", daily_signal_df["selected_call_signal"].sum())

display(daily_signal_df.tail(20))

Daily signal rows: 2011
Date range: 20180625 to 20260625

Signal state counts:


,count
signal_state,
none,766
both_put_and_call,584
put_only,496
call_only,165



Selected put signal dates: 1080
Selected call signal dates: 749


,trade_date,spx_close,spx_rsi_14,preferred_tenor,preferred_source,num_put_tenors,num_call_tenors,selected_put_signal,selected_put_tenor,selected_put_tier,...,selected_call_vol_ratio,selected_call_is_preferred_tenor,selected_call_short_strike_rule,both_put_and_call_signal,any_trade_signal,signal_state,provisional_put_risk_pct,provisional_call_risk_pct,provisional_calibrated_risk_pct,needs_call_risk_calibration
1991,20260528,7563.63,72.725224,12.0,interior_preferred,1,1,True,12.0,C_acceptable,...,1.540583,True,+2sd_otm_short_call,True,True,both_put_and_call,2.7,NaN,2.7,True
1992,20260529,7580.06,73.529728,9.0,endpoint_allowed,1,2,True,21.0,C_acceptable,...,2.080837,True,+2sd_otm_short_call,True,True,both_put_and_call,2.7,NaN,2.7,True
1993,20260601,7599.96,74.510414,12.0,interior_preferred,5,7,True,15.0,C_acceptable,...,2.558829,True,+2sd_otm_short_call,True,True,both_put_and_call,2.7,NaN,2.7,True
1994,20260602,7609.78,75.002578,12.0,interior_preferred,6,8,True,18.0,C_acceptable,...,2.546626,True,+2sd_otm_short_call,True,True,both_put_and_call,2.7,NaN,2.7,True
1995,20260603,7553.68,67.038959,12.0,interior_preferred,7,8,True,18.0,B_good,...,2.182300,True,+2sd_otm_short_call,True,True,both_put_and_call,3.2,NaN,3.2,True
1996,20260604,7584.31,68.975841,15.0,interior_preferred,6,7,True,12.0,B_good,...,2.147465,True,+2sd_otm_short_call,True,True,both_put_and_call,3.2,NaN,3.2,True
1997,20260605,7383.74,48.767312,30.0,interior_preferred,1,0,True,30.0,C_acceptable,...,NaN,False,,False,True,put_only,2.7,0.0,2.7,False
1998,20260608,7405.73,50.480320,24.0,interior_preferred,0,0,False,NaN,none,...,NaN,False,,False,False,none,0.0,0.0,0.0,False
1999,20260609,7386.65,48.950955,24.0,interior_preferred,4,0,True,24.0,C_acceptable,...,NaN,False,,False,True,put_only,2.7,0.0,2.7,False
2000,20260610,7266.99,40.636070,21.0,interior_preferred,6,0,True,21.0,C_acceptable,...,NaN,False,,False,True,put_only,2.7,0.0,2.7,False


In [3]:
# ============================================================
# Date and strike helper functions
# ============================================================

def int_yyyymmdd_to_timestamp(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def timestamp_to_int_yyyymmdd(ts):
    return int(pd.Timestamp(ts).strftime("%Y%m%d"))


def add_calendar_days_to_trade_date(trade_date, tenor_days):
    """
    Target expiry date before mapping to listed option expiries.
    """
    if pd.isna(tenor_days):
        return np.nan

    trade_ts = int_yyyymmdd_to_timestamp(trade_date)
    target_expiry_ts = trade_ts + pd.Timedelta(days=int(tenor_days))

    return timestamp_to_int_yyyymmdd(target_expiry_ts)


def raw_atm_put_strike(spx_close):
    """
    Raw naked short put strike:
    ATM rounded down.
    Later we will map this to nearest listed put strike <= SPX.
    """
    if pd.isna(spx_close):
        return np.nan

    return np.floor(float(spx_close))


def raw_2sd_call_strike(spx_close, implied_vol, tenor_days):
    """
    Raw naked short call strike:
    SPX * (1 + 2 * implied_vol_decimal * sqrt(T))

    Uses selected tenor for now.
    After mapping to actual listed expiry, we may recalculate using actual DTE.
    """
    if pd.isna(spx_close) or pd.isna(implied_vol) or pd.isna(tenor_days):
        return np.nan

    implied_vol_decimal = float(implied_vol) / 100.0
    t = float(tenor_days) / 365.0

    return float(spx_close) * (1.0 + 2.0 * implied_vol_decimal * np.sqrt(t))


# Quick tests
test_trade_date = 20260625
test_spx = 7357.49
test_tenor = 30
test_iv = 15.0

print("Test target expiry:", add_calendar_days_to_trade_date(test_trade_date, test_tenor))
print("Test raw put strike:", raw_atm_put_strike(test_spx))
print("Test raw 2SD call strike:", raw_2sd_call_strike(test_spx, test_iv, test_tenor))

Test target expiry: 20260725
Test raw put strike: 7357.0
Test raw 2SD call strike: 7990.288047314994


In [4]:
# ============================================================
# Candidate mapping assumptions
# ============================================================

UNDERLYER = "SPX"

PUT_OPTION_STRUCTURE = "naked_short_put"
CALL_OPTION_STRUCTURE = "naked_short_call"

PUT_OPTION_RIGHT = "P"
CALL_OPTION_RIGHT = "C"

PUT_SHORT_STRIKE_RULE = "atm_rounded_down_nearest_listed_put_strike_lte_spx"
CALL_SHORT_STRIKE_RULE = "plus_2sd_otm_nearest_listed_call_strike_gte_raw"

EXPIRY_SELECTION_RULE = "closest_listed_expiry_to_target_later_on_tie"

DEFAULT_EXIT_RULE = "hold_to_expiry"

print("Underlyer:", UNDERLYER)
print("Put structure:", PUT_OPTION_STRUCTURE)
print("Call structure:", CALL_OPTION_STRUCTURE)
print("Expiry selection:", EXPIRY_SELECTION_RULE)
print("Exit rule:", DEFAULT_EXIT_RULE)

Underlyer: SPX
Put structure: naked_short_put
Call structure: naked_short_call
Expiry selection: closest_listed_expiry_to_target_later_on_tie
Exit rule: hold_to_expiry


In [5]:
# ============================================================
# Build raw trade candidate rows
# ============================================================

candidate_rows = []

for _, row in daily_signal_df.iterrows():
    trade_date = row["trade_date"]
    spx_close = row["spx_close"]

    # ----------------------------
    # Naked short put candidate
    # ----------------------------
    if bool(row["selected_put_signal"]):
        selected_tenor = row["selected_put_tenor"]
        target_expiry_date = add_calendar_days_to_trade_date(
            trade_date=trade_date,
            tenor_days=selected_tenor,
        )

        short_strike_raw = raw_atm_put_strike(spx_close)

        candidate_rows.append({
            "trade_date": trade_date,
            "underlyer": UNDERLYER,
            "trade_side": "put",
            "option_structure": PUT_OPTION_STRUCTURE,
            "option_right": PUT_OPTION_RIGHT,

            "selected_tenor": selected_tenor,
            "target_expiry_date": target_expiry_date,
            "expiry_selection_rule": EXPIRY_SELECTION_RULE,
            "exit_rule": DEFAULT_EXIT_RULE,

            "short_strike_rule": PUT_SHORT_STRIKE_RULE,
            "short_strike_raw": short_strike_raw,

            # These will be populated after we inspect/list actual chains.
            "selected_expiry_date": np.nan,
            "actual_dte": np.nan,
            "short_strike_selected": np.nan,

            "signal_state": row["signal_state"],
            "signal_tier": row["selected_put_tier"],

            "signal_primary_vrp": row["selected_put_primary_vrp_signal"],
            "signal_blended_z": row["selected_put_blended_z"],
            "signal_3m_z": row["selected_put_3m_z"],
            "signal_1y_z": row["selected_put_1y_z"],

            "signal_implied_vol": row["selected_put_implied_vol"],
            "signal_trailing_rv": row["selected_put_trailing_rv"],
            "signal_variance_ratio": row["selected_put_variance_ratio"],
            "signal_vol_ratio": row["selected_put_vol_ratio"],

            "spx_close": spx_close,
            "spx_rsi_14": row["spx_rsi_14"],
            "preferred_tenor": row["preferred_tenor"],
        })

    # ----------------------------
    # Naked short call candidate
    # ----------------------------
    if bool(row["selected_call_signal"]):
        selected_tenor = row["selected_call_tenor"]
        target_expiry_date = add_calendar_days_to_trade_date(
            trade_date=trade_date,
            tenor_days=selected_tenor,
        )

        short_strike_raw = raw_2sd_call_strike(
            spx_close=spx_close,
            implied_vol=row["selected_call_implied_vol"],
            tenor_days=selected_tenor,
        )

        candidate_rows.append({
            "trade_date": trade_date,
            "underlyer": UNDERLYER,
            "trade_side": "call",
            "option_structure": CALL_OPTION_STRUCTURE,
            "option_right": CALL_OPTION_RIGHT,

            "selected_tenor": selected_tenor,
            "target_expiry_date": target_expiry_date,
            "expiry_selection_rule": EXPIRY_SELECTION_RULE,
            "exit_rule": DEFAULT_EXIT_RULE,

            "short_strike_rule": CALL_SHORT_STRIKE_RULE,
            "short_strike_raw": short_strike_raw,

            # These will be populated after we inspect/list actual chains.
            "selected_expiry_date": np.nan,
            "actual_dte": np.nan,
            "short_strike_selected": np.nan,

            "signal_state": row["signal_state"],
            "signal_tier": "call_baseline",

            "signal_primary_vrp": row["selected_call_primary_vrp_signal"],
            "signal_blended_z": row["selected_call_blended_z"],
            "signal_3m_z": row["selected_call_3m_z"],
            "signal_1y_z": row["selected_call_1y_z"],

            "signal_implied_vol": row["selected_call_implied_vol"],
            "signal_trailing_rv": row["selected_call_trailing_rv"],
            "signal_variance_ratio": row["selected_call_variance_ratio"],
            "signal_vol_ratio": row["selected_call_vol_ratio"],

            "spx_close": spx_close,
            "spx_rsi_14": row["spx_rsi_14"],
            "preferred_tenor": row["preferred_tenor"],
        })

trade_candidate_raw_df = pd.DataFrame(candidate_rows)

print("Raw candidate rows:", len(trade_candidate_raw_df))
print("Unique trade dates:", trade_candidate_raw_df["trade_date"].nunique())

print("\nCandidates by side:")
display(trade_candidate_raw_df["trade_side"].value_counts().rename("count").to_frame())

print("\nCandidates by signal state:")
display(trade_candidate_raw_df["signal_state"].value_counts().rename("count").to_frame())

print("\nCandidates by side and signal state:")
display(
    trade_candidate_raw_df
    .groupby(["signal_state", "trade_side"])
    .size()
    .unstack(fill_value=0)
)

display(trade_candidate_raw_df.tail(40))

Raw candidate rows: 1829
Unique trade dates: 1245

Candidates by side:


,count
trade_side,
put,1080
call,749



Candidates by signal state:


,count
signal_state,
both_put_and_call,1168
put_only,496
call_only,165



Candidates by side and signal state:


trade_side,call,put
signal_state,,
both_put_and_call,584,584
call_only,165,0
put_only,0,496


,trade_date,underlyer,trade_side,option_structure,option_right,selected_tenor,target_expiry_date,expiry_selection_rule,exit_rule,short_strike_rule,...,signal_blended_z,signal_3m_z,signal_1y_z,signal_implied_vol,signal_trailing_rv,signal_variance_ratio,signal_vol_ratio,spx_close,spx_rsi_14,preferred_tenor
1789,20260429,SPX,put,naked_short_put,P,12.0,20260511,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,0.858889,0.884639,0.833139,17.422249,8.900762,3.831370,1.957389,7135.95,66.585933,12.0
1790,20260429,SPX,call,naked_short_call,C,12.0,20260511,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,0.858889,0.884639,0.833139,17.422249,8.900762,3.831370,1.957389,7135.95,66.585933,12.0
1791,20260504,SPX,put,naked_short_put,P,12.0,20260516,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,0.868041,0.879808,0.856274,16.720195,8.451871,3.913605,1.978283,7200.75,67.863337,12.0
1792,20260504,SPX,call,naked_short_call,C,12.0,20260516,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,0.868041,0.879808,0.856274,16.720195,8.451871,3.913605,1.978283,7200.75,67.863337,12.0
1793,20260505,SPX,put,naked_short_put,P,18.0,20260523,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,0.532218,0.571118,0.493318,16.475916,9.602682,2.943839,1.715762,7259.22,70.785030,18.0
1794,20260505,SPX,call,naked_short_call,C,18.0,20260523,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,0.532218,0.571118,0.493318,16.475916,9.602682,2.943839,1.715762,7259.22,70.785030,18.0
1795,20260511,SPX,put,naked_short_put,P,24.0,20260604,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,0.443489,0.516379,0.370599,17.804969,10.700932,2.768466,1.663871,7412.84,75.115221,24.0
1796,20260511,SPX,call,naked_short_call,C,24.0,20260604,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,0.443489,0.516379,0.370599,17.804969,10.700932,2.768466,1.663871,7412.84,75.115221,24.0
1797,20260512,SPX,put,naked_short_put,P,24.0,20260605,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,atm_rounded_down_nearest_listed_put_strike_lte...,...,0.324947,0.394858,0.255037,17.317442,10.719197,2.610015,1.615554,7400.96,73.683428,24.0
1798,20260512,SPX,call,naked_short_call,C,24.0,20260605,closest_listed_expiry_to_target_later_on_tie,hold_to_expiry,plus_2sd_otm_nearest_listed_call_strike_gte_raw,...,0.324947,0.394858,0.255037,17.317442,10.719197,2.610015,1.615554,7400.96,73.683428,24.0


In [6]:
# ============================================================
# Save raw target trade candidates
# ============================================================

trade_candidate_raw_df.to_csv(TRADE_CANDIDATE_RAW_CSV, index=False)
trade_candidate_raw_df.to_parquet(TRADE_CANDIDATE_RAW_PARQUET, index=False)

print("Saved raw candidate CSV:", TRADE_CANDIDATE_RAW_CSV)
print("Saved raw candidate parquet:", TRADE_CANDIDATE_RAW_PARQUET)

Saved raw candidate CSV: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1_raw_targets.csv
Saved raw candidate parquet: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1_raw_targets.parquet


In [7]:
# ============================================================
# Stage 3: Inspect local ThetaData chain cache
# ============================================================

RAW_CHAIN_CACHE_DIR = RAW_DATA_DIR / "thetadata_chains"

print("Raw chain cache directory:", RAW_CHAIN_CACHE_DIR)
print("Exists:", RAW_CHAIN_CACHE_DIR.exists())

if not RAW_CHAIN_CACHE_DIR.exists():
    raise FileNotFoundError(f"Raw chain cache directory not found: {RAW_CHAIN_CACHE_DIR}")

chain_files = sorted([
    p for p in RAW_CHAIN_CACHE_DIR.rglob("*")
    if p.is_file()
])

print("Number of files in raw chain cache:", len(chain_files))

if len(chain_files) == 0:
    raise FileNotFoundError(f"No files found in {RAW_CHAIN_CACHE_DIR}")

print("\nFirst 20 files:")
for p in chain_files[:20]:
    print(p.relative_to(RAW_CHAIN_CACHE_DIR))

print("\nLast 20 files:")
for p in chain_files[-20:]:
    print(p.relative_to(RAW_CHAIN_CACHE_DIR))

Raw chain cache directory: C:\Users\patri\vrp_project\data\raw\thetadata_chains
Exists: True
Number of files in raw chain cache: 10901

First 20 files:
SPX_20180625_20180720_160000.pkl
SPX_20180626_20180720_160000.pkl
SPX_20180627_20180720_160000.pkl
SPX_20180628_20180720_160000.pkl
SPX_20180629_20180720_160000.pkl
SPX_20180702_20180720_160000.pkl
SPX_20180703_20180720_160000.pkl
SPX_20180705_20180720_160000.pkl
SPX_20180706_20180720_160000.pkl
SPX_20180709_20180720_160000.pkl
SPX_20180709_20180817_160000.pkl
SPX_20180710_20180720_160000.pkl
SPX_20180710_20180817_160000.pkl
SPX_20180711_20180720_160000.pkl
SPX_20180711_20180817_160000.pkl
SPX_20180712_20180720_160000.pkl
SPX_20180712_20180817_160000.pkl
SPX_20180713_20180720_160000.pkl
SPX_20180713_20180817_160000.pkl
SPX_20180716_20180720_160000.pkl

Last 20 files:
SPXW_20260618_20260702_160000.pkl
SPXW_20260618_20260710_160000.pkl
SPXW_20260618_20260724_160000.pkl
SPXW_20260622_20260626_160000.pkl
SPXW_20260622_20260702_160000.pkl
SP

In [8]:
# ============================================================
# Build cache file inventory
# ============================================================

file_inventory_rows = []

for p in chain_files:
    stat = p.stat()

    file_inventory_rows.append({
        "file_path": str(p),
        "relative_path": str(p.relative_to(RAW_CHAIN_CACHE_DIR)),
        "file_name": p.name,
        "suffix": p.suffix.lower(),
        "size_mb": stat.st_size / (1024 * 1024),
        "modified_time": pd.Timestamp(stat.st_mtime, unit="s"),
    })

cache_inventory_df = pd.DataFrame(file_inventory_rows)

print("Files:", len(cache_inventory_df))
print("\nSuffix counts:")
display(cache_inventory_df["suffix"].value_counts().rename("count").to_frame())

print("\nSize summary, MB:")
display(cache_inventory_df["size_mb"].describe())

display(cache_inventory_df.head(20))
display(cache_inventory_df.tail(20))

Files: 10901

Suffix counts:


,count
suffix,
.pkl,10901



Size summary, MB:


count    10901.000000
mean         0.065139
std          0.022406
min          0.016544
25%          0.048794
50%          0.058055
75%          0.079673
max          0.151597
Name: size_mb, dtype: float64

,file_path,relative_path,file_name,suffix,size_mb,modified_time
0,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180625_20180720_160000.pkl,SPX_20180625_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:11:34.556218863
1,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180626_20180720_160000.pkl,SPX_20180626_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:13:44.021209002
2,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180627_20180720_160000.pkl,SPX_20180627_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:14:02.647611618
3,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180628_20180720_160000.pkl,SPX_20180628_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:14:18.638312340
4,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180629_20180720_160000.pkl,SPX_20180629_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:14:34.915855408
5,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180702_20180720_160000.pkl,SPX_20180702_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:14:52.373704195
6,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180703_20180720_160000.pkl,SPX_20180703_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:15:08.931561470
7,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180705_20180720_160000.pkl,SPX_20180705_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:15:26.418432713
8,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180706_20180720_160000.pkl,SPX_20180706_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:15:43.197410583
9,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180709_20180720_160000.pkl,SPX_20180709_20180720_160000.pkl,.pkl,0.064201,2026-06-26 19:15:58.114251375


,file_path,relative_path,file_name,suffix,size_mb,modified_time
10881,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260618_20260702_160000.pkl,SPXW_20260618_20260702_160000.pkl,.pkl,0.064238,2026-06-27 21:04:20.044897318
10882,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260618_20260710_160000.pkl,SPXW_20260618_20260710_160000.pkl,.pkl,0.055205,2026-06-27 21:04:19.621849060
10883,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260618_20260724_160000.pkl,SPXW_20260618_20260724_160000.pkl,.pkl,0.042382,2026-06-27 21:04:24.552203655
10884,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260622_20260626_160000.pkl,SPXW_20260622_20260626_160000.pkl,.pkl,0.073974,2026-06-27 21:04:38.251928568
10885,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260622_20260702_160000.pkl,SPXW_20260622_20260702_160000.pkl,.pkl,0.064238,2026-06-27 21:04:38.607841969
10886,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260622_20260710_160000.pkl,SPXW_20260622_20260710_160000.pkl,.pkl,0.055205,2026-06-27 21:04:36.342964888
10887,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260622_20260724_160000.pkl,SPXW_20260622_20260724_160000.pkl,.pkl,0.042382,2026-06-27 21:04:40.697266817
10888,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260622_20260731_160000.pkl,SPXW_20260622_20260731_160000.pkl,.pkl,0.114107,2026-06-27 21:04:50.762918949
10889,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260623_20260702_160000.pkl,SPXW_20260623_20260702_160000.pkl,.pkl,0.064238,2026-06-27 21:05:04.633722067
10890,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260623_20260710_160000.pkl,SPXW_20260623_20260710_160000.pkl,.pkl,0.055205,2026-06-27 21:05:03.053114414


In [9]:
# ============================================================
# Infer date tokens from file paths
# ============================================================

import re

def extract_yyyymmdd_tokens(text):
    tokens = re.findall(r"(20\d{6})", str(text))
    valid_tokens = []

    for token in tokens:
        try:
            pd.to_datetime(token, format="%Y%m%d")
            valid_tokens.append(int(token))
        except Exception:
            pass

    return valid_tokens


cache_inventory_df["yyyymmdd_tokens"] = cache_inventory_df["relative_path"].apply(
    extract_yyyymmdd_tokens
)

cache_inventory_df["num_date_tokens"] = cache_inventory_df["yyyymmdd_tokens"].apply(len)

display(
    cache_inventory_df[
        [
            "relative_path",
            "suffix",
            "size_mb",
            "num_date_tokens",
            "yyyymmdd_tokens",
        ]
    ].head(50)
)

print("Date-token count distribution:")
display(cache_inventory_df["num_date_tokens"].value_counts().sort_index().rename("count").to_frame())

# If there are at least two date tokens, often the first is trade date and second is expiry date.
cache_inventory_df["inferred_trade_date_from_path"] = cache_inventory_df["yyyymmdd_tokens"].apply(
    lambda x: x[0] if len(x) >= 1 else np.nan
)

cache_inventory_df["inferred_expiry_date_from_path"] = cache_inventory_df["yyyymmdd_tokens"].apply(
    lambda x: x[1] if len(x) >= 2 else np.nan
)

print("Inferred trade date range from path:")
print(
    cache_inventory_df["inferred_trade_date_from_path"].min(),
    "to",
    cache_inventory_df["inferred_trade_date_from_path"].max(),
)

print("Inferred expiry date range from path:")
print(
    cache_inventory_df["inferred_expiry_date_from_path"].min(),
    "to",
    cache_inventory_df["inferred_expiry_date_from_path"].max(),
)

,relative_path,suffix,size_mb,num_date_tokens,yyyymmdd_tokens
0,SPX_20180625_20180720_160000.pkl,.pkl,0.064201,2,"[20180625, 20180720]"
1,SPX_20180626_20180720_160000.pkl,.pkl,0.064201,2,"[20180626, 20180720]"
2,SPX_20180627_20180720_160000.pkl,.pkl,0.064201,2,"[20180627, 20180720]"
3,SPX_20180628_20180720_160000.pkl,.pkl,0.064201,2,"[20180628, 20180720]"
4,SPX_20180629_20180720_160000.pkl,.pkl,0.064201,2,"[20180629, 20180720]"
5,SPX_20180702_20180720_160000.pkl,.pkl,0.064201,2,"[20180702, 20180720]"
6,SPX_20180703_20180720_160000.pkl,.pkl,0.064201,2,"[20180703, 20180720]"
7,SPX_20180705_20180720_160000.pkl,.pkl,0.064201,2,"[20180705, 20180720]"
8,SPX_20180706_20180720_160000.pkl,.pkl,0.064201,2,"[20180706, 20180720]"
9,SPX_20180709_20180720_160000.pkl,.pkl,0.064201,2,"[20180709, 20180720]"


Date-token count distribution:


,count
num_date_tokens,
2,10901


Inferred trade date range from path:
20180625 to 20260625
Inferred expiry date range from path:
20180629 to 20260731


In [11]:
# ============================================================
# Read one sample cached chain file
# ============================================================

def convert_pickle_object_to_dataframe(obj):
    """
    Convert a pickle-loaded object into a DataFrame when possible.
    """
    if isinstance(obj, pd.DataFrame):
        return obj

    if isinstance(obj, dict):
        # If the dict contains one or more DataFrames, use the largest one.
        dataframe_items = {
            k: v for k, v in obj.items()
            if isinstance(v, pd.DataFrame)
        }

        if len(dataframe_items) > 0:
            largest_key = max(dataframe_items, key=lambda k: len(dataframe_items[k]))
            print("Pickle object is dict. Using DataFrame key:", largest_key)
            return dataframe_items[largest_key]

        # Otherwise try direct DataFrame conversion.
        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle dict could not be converted to DataFrame: {e}")

    if isinstance(obj, list):
        try:
            return pd.DataFrame(obj)
        except Exception as e:
            raise ValueError(f"Pickle list could not be converted to DataFrame: {e}")

    raise ValueError(f"Unsupported pickle object type: {type(obj)}")


def read_cached_chain_file(file_path, nrows=None):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".parquet":
        df = pd.read_parquet(file_path)

    elif suffix == ".csv":
        df = pd.read_csv(file_path)

    elif suffix in [".txt", ".tsv"]:
        df = pd.read_csv(file_path)

    elif suffix == ".pkl":
        obj = pd.read_pickle(file_path)
        df = convert_pickle_object_to_dataframe(obj)

    else:
        raise ValueError(f"Unsupported cached chain file suffix: {suffix}")

    if nrows is not None:
        df = df.head(nrows)

    return df


# Prefer a non-empty file
sample_file_row = (
    cache_inventory_df[cache_inventory_df["size_mb"] > 0]
    .sort_values("size_mb", ascending=False)
    .head(1)
)

if sample_file_row.empty:
    raise ValueError("No non-empty cached chain files found.")

sample_file_path = sample_file_row.iloc[0]["file_path"]

print("Sample file:", sample_file_path)

sample_chain_df = read_cached_chain_file(sample_file_path)

print("Sample rows:", len(sample_chain_df))
print("Sample columns:")
for c in sample_chain_df.columns:
    print(" ", c)

print("\nDtypes:")
display(sample_chain_df.dtypes.rename("dtype").to_frame())

display(sample_chain_df.head(20))

Sample file: C:\Users\patri\vrp_project\data\raw\thetadata_chains\SPX_20260615_20260618_160000.pkl
Sample rows: 1278
Sample columns:
  root
  expiration
  strike
  right
  bid
  ask
  mid
  bid_size
  ask_size
  bid_exchange
  ask_exchange
  bid_condition
  ask_condition
  timestamp

Dtypes:


,dtype
root,object
expiration,int64
strike,float64
right,object
bid,float64
ask,float64
mid,float64
bid_size,int64
ask_size,int64
bid_exchange,int64


,root,expiration,strike,right,bid,ask,mid,bid_size,ask_size,bid_exchange,ask_exchange,bid_condition,ask_condition,timestamp
0,SPX,20260618,6390.0,C,1163.20,1180.60,1171.900,1,1,5,5,50,50,2026-06-15T16:00:00.000
1,SPX,20260618,6390.0,P,0.10,0.35,0.225,58,71,5,5,50,50,2026-06-15T16:00:00.000
2,SPX,20260618,8010.0,P,438.40,456.10,447.250,1,1,5,5,50,50,2026-06-15T16:00:00.000
3,SPX,20260618,8010.0,C,0.00,0.75,0.375,0,20,5,5,50,50,2026-06-15T16:00:00.000
4,SPX,20260618,7285.0,P,2.80,2.95,2.875,12,10,5,5,50,50,2026-06-15T16:00:00.000
5,SPX,20260618,7840.0,C,0.05,0.45,0.250,69,20,5,5,50,50,2026-06-15T16:00:00.000
6,SPX,20260618,7285.0,C,271.70,289.10,280.400,2,3,5,5,50,50,2026-06-15T16:00:00.000
7,SPX,20260618,7840.0,P,268.60,285.90,277.250,1,1,5,5,50,50,2026-06-15T16:00:00.000
8,SPX,20260618,6220.0,P,0.10,0.20,0.150,66,10,5,5,50,50,2026-06-15T16:00:00.000
9,SPX,20260618,6220.0,C,1333.30,1352.60,1342.950,1,1,5,5,50,50,2026-06-15T16:00:00.000


In [12]:
# ============================================================
# Inspect columns across multiple cached files
# ============================================================

sample_files_for_schema = (
    cache_inventory_df[cache_inventory_df["size_mb"] > 0]
    .sort_values("file_name")
    .head(20)
)

schema_rows = []

for _, row in sample_files_for_schema.iterrows():
    p = row["file_path"]

    try:
        df_sample = read_cached_chain_file(p, nrows=5)

        schema_rows.append({
            "relative_path": row["relative_path"],
            "suffix": row["suffix"],
            "num_columns": len(df_sample.columns),
            "columns": list(df_sample.columns),
            "read_ok": True,
            "error": "",
        })

    except Exception as e:
        schema_rows.append({
            "relative_path": row["relative_path"],
            "suffix": row["suffix"],
            "num_columns": np.nan,
            "columns": [],
            "read_ok": False,
            "error": str(e),
        })

schema_inspection_df = pd.DataFrame(schema_rows)

display(schema_inspection_df)

print("Read OK counts:")
display(schema_inspection_df["read_ok"].value_counts().rename("count").to_frame())

,relative_path,suffix,num_columns,columns,read_ok,error
0,SPXW_20180625_20180629_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
1,SPXW_20180625_20180706_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
2,SPXW_20180625_20180713_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
3,SPXW_20180625_20180727_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
4,SPXW_20180625_20180803_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
5,SPXW_20180626_20180629_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
6,SPXW_20180626_20180706_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
7,SPXW_20180626_20180713_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
8,SPXW_20180626_20180727_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,
9,SPXW_20180626_20180803_160000.pkl,.pkl,14,"[root, expiration, strike, right, bid, ask, mi...",True,


Read OK counts:


,count
read_ok,
True,20


In [13]:
# ============================================================
# Identify likely option-chain columns
# ============================================================

all_columns = list(sample_chain_df.columns)
lower_col_map = {str(c).lower(): c for c in all_columns}

def find_first_existing_column(possible_names):
    for name in possible_names:
        if name.lower() in lower_col_map:
            return lower_col_map[name.lower()]
    return None


LIKELY_TRADE_DATE_COL = find_first_existing_column([
    "trade_date",
    "date",
    "quote_date",
    "ms_of_day",
])

LIKELY_EXPIRY_COL = find_first_existing_column([
    "expiration",
    "expiration_date",
    "expiry",
    "expiry_date",
    "exp",
])

LIKELY_STRIKE_COL = find_first_existing_column([
    "strike",
    "strike_price",
])

LIKELY_RIGHT_COL = find_first_existing_column([
    "right",
    "option_type",
    "put_call",
    "cp",
])

LIKELY_BID_COL = find_first_existing_column([
    "bid",
])

LIKELY_ASK_COL = find_first_existing_column([
    "ask",
])

print("Likely trade/date column:", LIKELY_TRADE_DATE_COL)
print("Likely expiry column:", LIKELY_EXPIRY_COL)
print("Likely strike column:", LIKELY_STRIKE_COL)
print("Likely right column:", LIKELY_RIGHT_COL)
print("Likely bid column:", LIKELY_BID_COL)
print("Likely ask column:", LIKELY_ASK_COL)

if LIKELY_EXPIRY_COL is not None:
    print("\nExpiry sample:")
    display(sample_chain_df[LIKELY_EXPIRY_COL].drop_duplicates().head(20))

if LIKELY_STRIKE_COL is not None:
    print("\nStrike sample:")
    display(sample_chain_df[LIKELY_STRIKE_COL].drop_duplicates().sort_values().head(20))

if LIKELY_RIGHT_COL is not None:
    print("\nRight sample:")
    display(sample_chain_df[LIKELY_RIGHT_COL].drop_duplicates().head(20))

Likely trade/date column: None
Likely expiry column: expiration
Likely strike column: strike
Likely right column: right
Likely bid column: bid
Likely ask column: ask

Expiry sample:


0    20260618
Name: expiration, dtype: int64


Strike sample:


772      200.0
334      400.0
215      600.0
774      800.0
568     1000.0
232     1200.0
660     1400.0
620     1600.0
1064    1800.0
700     2000.0
510     2200.0
50      2300.0
1114    2400.0
1126    2500.0
914     2600.0
460     2700.0
528     2800.0
264     2900.0
955     3000.0
1172    3100.0
Name: strike, dtype: float64


Right sample:


0    C
1    P
Name: right, dtype: object

In [14]:
# ============================================================
# Stage 4: Build ThetaData cache index from filenames
# ============================================================

import re

# Recreate chain_files if needed
if "chain_files" not in globals():
    chain_files = sorted([
        p for p in RAW_CHAIN_CACHE_DIR.rglob("*")
        if p.is_file()
    ])

def parse_chain_cache_filename(path):
    """
    Expected pattern:
        SPX_20260615_20260618_160000.pkl

    Returns:
        symbol, trade_date, expiration, quote_time
    """
    path = Path(path)
    name = path.name

    pattern = r"^(?P<symbol>[A-Za-z0-9]+)_(?P<trade_date>\d{8})_(?P<expiration>\d{8})_(?P<quote_time>\d{6})\.pkl$"
    match = re.match(pattern, name)

    if match is None:
        return {
            "parse_ok": False,
            "symbol": None,
            "trade_date": np.nan,
            "expiration": np.nan,
            "quote_time": None,
            "parse_error": "filename_does_not_match_expected_pattern",
        }

    return {
        "parse_ok": True,
        "symbol": match.group("symbol"),
        "trade_date": int(match.group("trade_date")),
        "expiration": int(match.group("expiration")),
        "quote_time": match.group("quote_time"),
        "parse_error": "",
    }


cache_index_rows = []

for p in chain_files:
    parsed = parse_chain_cache_filename(p)
    stat = p.stat()

    cache_index_rows.append({
        "file_path": str(p),
        "file_name": p.name,
        "suffix": p.suffix.lower(),
        "size_mb": stat.st_size / (1024 * 1024),
        **parsed,
    })

cache_index_df = pd.DataFrame(cache_index_rows)

print("Cache files:", len(cache_index_df))
print("Parsed OK:", cache_index_df["parse_ok"].sum())
print("Parse failures:", (~cache_index_df["parse_ok"]).sum())

print("\nSymbol counts:")
display(cache_index_df["symbol"].value_counts(dropna=False).rename("count").to_frame())

print("\nTrade date range:")
print(cache_index_df["trade_date"].min(), "to", cache_index_df["trade_date"].max())

print("\nExpiration range:")
print(cache_index_df["expiration"].min(), "to", cache_index_df["expiration"].max())

print("\nQuote-time counts:")
display(cache_index_df["quote_time"].value_counts(dropna=False).rename("count").to_frame())

display(cache_index_df.head())
display(cache_index_df.tail())

Cache files: 10901
Parsed OK: 10901
Parse failures: 0

Symbol counts:


,count
symbol,
SPXW,8383
SPX,2518



Trade date range:
20180625 to 20260625

Expiration range:
20180629 to 20260731

Quote-time counts:


,count
quote_time,
160000,10772
130000,64
155900,27
155800,16
155500,11
155000,11


,file_path,file_name,suffix,size_mb,parse_ok,symbol,trade_date,expiration,quote_time,parse_error
0,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180625_20180720_160000.pkl,.pkl,0.064201,True,SPX,20180625,20180720,160000,
1,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180626_20180720_160000.pkl,.pkl,0.064201,True,SPX,20180626,20180720,160000,
2,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180627_20180720_160000.pkl,.pkl,0.064201,True,SPX,20180627,20180720,160000,
3,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180628_20180720_160000.pkl,.pkl,0.064201,True,SPX,20180628,20180720,160000,
4,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPX_20180629_20180720_160000.pkl,.pkl,0.064201,True,SPX,20180629,20180720,160000,


,file_path,file_name,suffix,size_mb,parse_ok,symbol,trade_date,expiration,quote_time,parse_error
10896,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260624_20260731_160000.pkl,.pkl,0.114107,True,SPXW,20260624,20260731,160000,
10897,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260625_20260702_160000.pkl,.pkl,0.066850,True,SPXW,20260625,20260702,160000,
10898,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260625_20260710_160000.pkl,.pkl,0.057817,True,SPXW,20260625,20260710,160000,
10899,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260625_20260724_160000.pkl,.pkl,0.044519,True,SPXW,20260625,20260724,160000,
10900,C:\Users\patri\vrp_project\data\raw\thetadata_...,SPXW_20260625_20260731_160000.pkl,.pkl,0.114582,True,SPXW,20260625,20260731,160000,


In [15]:
# ============================================================
# Expiry mapping helpers
# ============================================================

def yyyymmdd_int_to_ts(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def calendar_day_diff(later_date_int, earlier_date_int):
    later_ts = yyyymmdd_int_to_ts(later_date_int)
    earlier_ts = yyyymmdd_int_to_ts(earlier_date_int)
    return int((later_ts - earlier_ts).days)


def select_closest_cached_expiry_for_trade_date(
    cache_index,
    trade_date,
    target_expiry_date,
):
    """
    Select listed expiry closest to target_expiry_date for a given trade_date.
    Tie-break: choose later expiry.
    """
    available = cache_index[
        (cache_index["parse_ok"])
        & (cache_index["trade_date"] == int(trade_date))
        & (cache_index["size_mb"] > 0)
    ].copy()

    if available.empty:
        return {
            "selected_expiry_date": np.nan,
            "selected_chain_file": "",
            "selected_symbol": "",
            "selected_quote_time": "",
            "expiry_distance_days": np.nan,
            "num_available_expiries_for_trade_date": 0,
            "expiry_mapping_status": "no_cached_chain_for_trade_date",
        }

    available["expiry_distance_days"] = available["expiration"].apply(
        lambda x: abs(calendar_day_diff(x, target_expiry_date))
    )

    # Tie-break: distance ascending, expiration descending.
    # If duplicate files for same expiry, prefer larger file.
    available = available.sort_values(
        ["expiry_distance_days", "expiration", "size_mb"],
        ascending=[True, False, False],
    )

    selected = available.iloc[0]

    return {
        "selected_expiry_date": int(selected["expiration"]),
        "selected_chain_file": selected["file_path"],
        "selected_symbol": selected["symbol"],
        "selected_quote_time": selected["quote_time"],
        "expiry_distance_days": int(selected["expiry_distance_days"]),
        "num_available_expiries_for_trade_date": available["expiration"].nunique(),
        "expiry_mapping_status": "mapped",
    }

In [16]:
# ============================================================
# Expiry mapping helpers
# ============================================================

def yyyymmdd_int_to_ts(date_int):
    return pd.to_datetime(str(int(date_int)), format="%Y%m%d")


def calendar_day_diff(later_date_int, earlier_date_int):
    later_ts = yyyymmdd_int_to_ts(later_date_int)
    earlier_ts = yyyymmdd_int_to_ts(earlier_date_int)
    return int((later_ts - earlier_ts).days)


def select_closest_cached_expiry_for_trade_date(
    cache_index,
    trade_date,
    target_expiry_date,
):
    """
    Select listed expiry closest to target_expiry_date for a given trade_date.
    Tie-break: choose later expiry.
    """
    available = cache_index[
        (cache_index["parse_ok"])
        & (cache_index["trade_date"] == int(trade_date))
        & (cache_index["size_mb"] > 0)
    ].copy()

    if available.empty:
        return {
            "selected_expiry_date": np.nan,
            "selected_chain_file": "",
            "selected_symbol": "",
            "selected_quote_time": "",
            "expiry_distance_days": np.nan,
            "num_available_expiries_for_trade_date": 0,
            "expiry_mapping_status": "no_cached_chain_for_trade_date",
        }

    available["expiry_distance_days"] = available["expiration"].apply(
        lambda x: abs(calendar_day_diff(x, target_expiry_date))
    )

    # Tie-break: distance ascending, expiration descending.
    # If duplicate files for same expiry, prefer larger file.
    available = available.sort_values(
        ["expiry_distance_days", "expiration", "size_mb"],
        ascending=[True, False, False],
    )

    selected = available.iloc[0]

    return {
        "selected_expiry_date": int(selected["expiration"]),
        "selected_chain_file": selected["file_path"],
        "selected_symbol": selected["symbol"],
        "selected_quote_time": selected["quote_time"],
        "expiry_distance_days": int(selected["expiry_distance_days"]),
        "num_available_expiries_for_trade_date": available["expiration"].nunique(),
        "expiry_mapping_status": "mapped",
    }

In [18]:
# ============================================================
# Apply expiry mapping to raw trade candidates
# ============================================================

# Reload raw candidates if needed
if "trade_candidate_raw_df" not in globals():
    trade_candidate_raw_df = pd.read_parquet(TRADE_CANDIDATE_RAW_PARQUET).copy()

expiry_mapping_rows = []

for _, row in trade_candidate_raw_df.iterrows():
    mapping = select_closest_cached_expiry_for_trade_date(
        cache_index=cache_index_df,
        trade_date=row["trade_date"],
        target_expiry_date=row["target_expiry_date"],
    )

    expiry_mapping_rows.append(mapping)

expiry_mapping_df = pd.DataFrame(expiry_mapping_rows)

# Drop placeholder columns before joining real mapped values
placeholder_cols_to_drop = [
    "selected_expiry_date",
    "actual_dte",
    "short_strike_selected",
]

trade_candidate_base_df = trade_candidate_raw_df.drop(
    columns=[c for c in placeholder_cols_to_drop if c in trade_candidate_raw_df.columns]
).copy()

trade_candidate_mapped_df = pd.concat(
    [
        trade_candidate_base_df.reset_index(drop=True),
        expiry_mapping_df.reset_index(drop=True),
    ],
    axis=1,
)

# Calculate actual DTE only for mapped rows
trade_candidate_mapped_df["actual_dte"] = np.nan

mapped_mask = trade_candidate_mapped_df["expiry_mapping_status"] == "mapped"

trade_candidate_mapped_df.loc[mapped_mask, "actual_dte"] = (
    trade_candidate_mapped_df.loc[mapped_mask]
    .apply(
        lambda x: calendar_day_diff(
            x["selected_expiry_date"],
            x["trade_date"],
        ),
        axis=1,
    )
)

print("Rows:", len(trade_candidate_mapped_df))

print("\nExpiry mapping status:")
display(
    trade_candidate_mapped_df["expiry_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nExpiry distance days:")
display(trade_candidate_mapped_df["expiry_distance_days"].describe())

print("\nActual DTE by side:")
display(
    trade_candidate_mapped_df
    .groupby("trade_side")["actual_dte"]
    .describe()
)

display(
    trade_candidate_mapped_df[
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "expiry_distance_days",
            "actual_dte",
            "selected_chain_file",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw",
        ]
    ].tail(40)
)

Rows: 1829

Expiry mapping status:


,count
expiry_mapping_status,
mapped,1829



Expiry distance days:


count    1829.000000
mean        1.610169
std         0.979285
min         0.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         4.000000
Name: expiry_distance_days, dtype: float64


Actual DTE by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,20.691589,8.048629,6.0,14.0,21.0,29.0,37.0
put,1080.0,22.868519,8.086381,7.0,16.0,24.0,30.0,37.0


,trade_date,trade_side,selected_tenor,target_expiry_date,selected_expiry_date,expiry_distance_days,actual_dte,selected_chain_file,spx_close,signal_implied_vol,short_strike_raw
1789,20260429,put,12.0,20260511,20260508,3,9.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7135.95,17.422249,7135.000000
1790,20260429,call,12.0,20260511,20260508,3,9.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7135.95,17.422249,7586.798062
1791,20260504,put,12.0,20260516,20260515,1,11.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7200.75,16.720195,7200.000000
1792,20260504,call,12.0,20260516,20260515,1,11.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7200.75,16.720195,7637.359583
1793,20260505,put,18.0,20260523,20260522,1,17.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7259.22,16.475916,7259.000000
1794,20260505,call,18.0,20260523,20260522,1,17.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7259.22,16.475916,7790.421578
1795,20260511,put,24.0,20260604,20260605,1,25.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7412.84,17.804969,7412.000000
1796,20260511,call,24.0,20260604,20260605,1,25.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7412.84,17.804969,8089.725207
1797,20260512,put,24.0,20260605,20260605,0,24.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7400.96,17.317442,7400.000000
1798,20260512,call,24.0,20260605,20260605,0,24.0,C:\Users\patri\vrp_project\data\raw\thetadata_...,7400.96,17.317442,8058.255984


In [19]:
# ============================================================
# Strike mapping helpers
# ============================================================

CHAIN_DF_CACHE = {}

def get_cached_chain_df(file_path):
    """
    Read a cached chain file once and reuse it.
    """
    if file_path in CHAIN_DF_CACHE:
        return CHAIN_DF_CACHE[file_path]

    df = read_cached_chain_file(file_path).copy()
    CHAIN_DF_CACHE[file_path] = df

    return df


def raw_2sd_call_strike_actual_dte(spx_close, implied_vol, actual_dte):
    """
    Recalculate call strike using actual listed expiry DTE.
    """
    if pd.isna(spx_close) or pd.isna(implied_vol) or pd.isna(actual_dte):
        return np.nan

    implied_vol_decimal = float(implied_vol) / 100.0
    t = float(actual_dte) / 365.0

    return float(spx_close) * (1.0 + 2.0 * implied_vol_decimal * np.sqrt(t))


def select_listed_short_strike_for_candidate(row):
    """
    For puts:
        selected strike = highest listed put strike <= floor(SPX)

    For calls:
        selected strike = lowest listed call strike >= raw 2SD call strike
        using actual DTE.
    """
    if row["expiry_mapping_status"] != "mapped":
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "skipped_due_to_unmapped_expiry",
        }

    file_path = row["selected_chain_file"]
    option_right = row["option_right"]

    try:
        chain_df = get_cached_chain_df(file_path)
    except Exception as e:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": f"chain_read_error: {e}",
        }

    required_cols = ["right", "strike"]

    missing_cols = [c for c in required_cols if c not in chain_df.columns]
    if missing_cols:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": f"missing_chain_columns: {missing_cols}",
        }

    right_chain_df = chain_df[
        chain_df["right"].astype(str).str.upper() == str(option_right).upper()
    ].copy()

    if right_chain_df.empty:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "no_strikes_for_option_right",
        }

    strikes = (
        pd.to_numeric(right_chain_df["strike"], errors="coerce")
        .dropna()
        .drop_duplicates()
        .sort_values()
        .to_numpy()
    )

    if len(strikes) == 0:
        return {
            "short_strike_raw_actual_dte": np.nan,
            "short_strike_selected": np.nan,
            "strike_mapping_status": "no_numeric_strikes_for_option_right",
        }

    if row["trade_side"] == "put":
        raw_strike = raw_atm_put_strike(row["spx_close"])

        eligible = strikes[strikes <= raw_strike]

        if len(eligible) == 0:
            return {
                "short_strike_raw_actual_dte": raw_strike,
                "short_strike_selected": np.nan,
                "strike_mapping_status": "no_put_strike_lte_raw",
            }

        selected_strike = eligible.max()

        return {
            "short_strike_raw_actual_dte": raw_strike,
            "short_strike_selected": selected_strike,
            "strike_mapping_status": "mapped",
        }

    if row["trade_side"] == "call":
        raw_strike = raw_2sd_call_strike_actual_dte(
            spx_close=row["spx_close"],
            implied_vol=row["signal_implied_vol"],
            actual_dte=row["actual_dte"],
        )

        eligible = strikes[strikes >= raw_strike]

        if len(eligible) == 0:
            return {
                "short_strike_raw_actual_dte": raw_strike,
                "short_strike_selected": np.nan,
                "strike_mapping_status": "no_call_strike_gte_raw",
            }

        selected_strike = eligible.min()

        return {
            "short_strike_raw_actual_dte": raw_strike,
            "short_strike_selected": selected_strike,
            "strike_mapping_status": "mapped",
        }

    return {
        "short_strike_raw_actual_dte": np.nan,
        "short_strike_selected": np.nan,
        "strike_mapping_status": "unknown_trade_side",
    }

In [20]:
# ============================================================
# Apply strike mapping
# ============================================================

strike_mapping_rows = []

for i, row in trade_candidate_mapped_df.iterrows():
    if i % 250 == 0:
        print(f"Processing strike mapping {i}/{len(trade_candidate_mapped_df)}")

    strike_mapping_rows.append(
        select_listed_short_strike_for_candidate(row)
    )

strike_mapping_df = pd.DataFrame(strike_mapping_rows)

trade_candidate_final_df = pd.concat(
    [
        trade_candidate_mapped_df.reset_index(drop=True),
        strike_mapping_df.reset_index(drop=True),
    ],
    axis=1,
)

# For downstream clarity, use actual-DTE raw strike as final raw strike.
trade_candidate_final_df["short_strike_raw_target_tenor"] = (
    trade_candidate_final_df["short_strike_raw"]
)

trade_candidate_final_df["short_strike_raw"] = (
    trade_candidate_final_df["short_strike_raw_actual_dte"]
)

trade_candidate_final_df["mapping_status"] = np.where(
    (trade_candidate_final_df["expiry_mapping_status"] == "mapped")
    & (trade_candidate_final_df["strike_mapping_status"] == "mapped"),
    "mapped",
    "not_mapped",
)

print("Rows:", len(trade_candidate_final_df))

print("\nOverall mapping status:")
display(
    trade_candidate_final_df["mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nExpiry mapping status:")
display(
    trade_candidate_final_df["expiry_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nStrike mapping status:")
display(
    trade_candidate_final_df["strike_mapping_status"]
    .value_counts()
    .rename("count")
    .to_frame()
)

print("\nMapped candidates by side:")
display(
    trade_candidate_final_df
    .groupby(["trade_side", "mapping_status"])
    .size()
    .unstack(fill_value=0)
)

display(
    trade_candidate_final_df[
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw_target_tenor",
            "short_strike_raw",
            "short_strike_selected",
            "mapping_status",
            "expiry_mapping_status",
            "strike_mapping_status",
        ]
    ].tail(50)
)

Processing strike mapping 0/1829
Processing strike mapping 250/1829
Processing strike mapping 500/1829
Processing strike mapping 750/1829
Processing strike mapping 1000/1829
Processing strike mapping 1250/1829
Processing strike mapping 1500/1829
Processing strike mapping 1750/1829
Rows: 1829

Overall mapping status:


,count
mapping_status,
mapped,1829



Expiry mapping status:


,count
expiry_mapping_status,
mapped,1829



Strike mapping status:


,count
strike_mapping_status,
mapped,1829



Mapped candidates by side:


mapping_status,mapped
trade_side,
call,749
put,1080


,trade_date,trade_side,selected_tenor,target_expiry_date,selected_expiry_date,actual_dte,spx_close,signal_implied_vol,short_strike_raw_target_tenor,short_strike_raw,short_strike_selected,mapping_status,expiry_mapping_status,strike_mapping_status
1779,20260325,put,30.0,20260424,20260424,30.0,6591.90,25.299814,6591.000000,6591.000000,6590.0,mapped,mapped,mapped
1780,20260326,put,30.0,20260425,20260424,29.0,6477.16,28.248942,6477.000000,6477.000000,6475.0,mapped,mapped,mapped
1781,20260327,put,30.0,20260426,20260424,28.0,6368.85,30.888215,6368.000000,6368.000000,6360.0,mapped,mapped,mapped
1782,20260330,put,30.0,20260429,20260501,32.0,6343.72,30.339952,6343.000000,6343.000000,6340.0,mapped,mapped,mapped
1783,20260421,put,12.0,20260503,20260501,10.0,7064.01,19.974643,7064.000000,7064.000000,7060.0,mapped,mapped,mapped
1784,20260421,call,12.0,20260503,20260501,10.0,7064.01,19.974643,7575.697163,7531.114336,7550.0,mapped,mapped,mapped
1785,20260427,put,9.0,20260506,20260508,11.0,7173.91,16.936112,7173.000000,7173.000000,7170.0,mapped,mapped,mapped
1786,20260427,call,9.0,20260506,20260508,11.0,7173.91,16.936112,7555.480215,7595.751745,7600.0,mapped,mapped,mapped
1787,20260428,put,9.0,20260507,20260508,10.0,7138.80,16.591230,7138.000000,7138.000000,7135.0,mapped,mapped,mapped
1788,20260428,call,9.0,20260507,20260508,10.0,7138.80,16.591230,7510.770593,7530.891433,7550.0,mapped,mapped,mapped


In [21]:
# ============================================================
# Save final mapped trade candidate panel
# ============================================================

TRADE_CANDIDATE_PANEL_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.csv"
TRADE_CANDIDATE_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.parquet"

TRADE_CANDIDATE_MAPPING_AUDIT_CSV = AUDIT_DIR / "trade_candidate_mapping_audit_v0_1.csv"

trade_candidate_final_df.to_csv(TRADE_CANDIDATE_PANEL_CSV, index=False)
trade_candidate_final_df.to_parquet(TRADE_CANDIDATE_PANEL_PARQUET, index=False)

mapping_audit_df = (
    trade_candidate_final_df
    .groupby(
        [
            "trade_side",
            "mapping_status",
            "expiry_mapping_status",
            "strike_mapping_status",
        ]
    )
    .size()
    .reset_index(name="count")
)

mapping_audit_df.to_csv(TRADE_CANDIDATE_MAPPING_AUDIT_CSV, index=False)

print("Saved trade candidate CSV:", TRADE_CANDIDATE_PANEL_CSV)
print("Saved trade candidate parquet:", TRADE_CANDIDATE_PANEL_PARQUET)
print("Saved mapping audit:", TRADE_CANDIDATE_MAPPING_AUDIT_CSV)

display(mapping_audit_df)

Saved trade candidate CSV: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1.csv
Saved trade candidate parquet: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1.parquet
Saved mapping audit: C:\Users\patri\vrp_project\data\audit\trade_candidate_mapping_audit_v0_1.csv


,trade_side,mapping_status,expiry_mapping_status,strike_mapping_status,count
0,call,mapped,mapped,mapped,749
1,put,mapped,mapped,mapped,1080


In [22]:
# ============================================================
# Final QA: expiry and DTE mapping
# ============================================================

trade_candidate_final_df["dte_minus_selected_tenor"] = (
    trade_candidate_final_df["actual_dte"]
    - trade_candidate_final_df["selected_tenor"]
)

print("Actual DTE minus selected tenor:")
display(trade_candidate_final_df["dte_minus_selected_tenor"].describe())

print("\nExpiry distance days:")
display(trade_candidate_final_df["expiry_distance_days"].describe())

print("\nDTE difference by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["dte_minus_selected_tenor"]
    .describe()
)

print("\nLargest absolute DTE differences:")
display(
    trade_candidate_final_df
    .assign(abs_dte_diff=lambda x: x["dte_minus_selected_tenor"].abs())
    .sort_values("abs_dte_diff", ascending=False)
    [
        [
            "trade_date",
            "trade_side",
            "selected_tenor",
            "target_expiry_date",
            "selected_expiry_date",
            "actual_dte",
            "dte_minus_selected_tenor",
            "expiry_distance_days",
        ]
    ]
    .head(30)
)

Actual DTE minus selected tenor:


count    1829.000000
mean       -0.144888
std         1.879377
min        -3.000000
25%        -2.000000
50%         0.000000
75%         1.000000
max         4.000000
Name: dte_minus_selected_tenor, dtype: float64


Expiry distance days:


count    1829.000000
mean        1.610169
std         0.979285
min         0.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         4.000000
Name: expiry_distance_days, dtype: float64


DTE difference by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,-0.148198,1.833493,-3.0,-2.0,0.0,1.0,4.0
put,1080.0,-0.142593,1.911392,-3.0,-2.0,0.0,1.0,4.0



Largest absolute DTE differences:


,trade_date,trade_side,selected_tenor,target_expiry_date,selected_expiry_date,actual_dte,dte_minus_selected_tenor,expiry_distance_days
853,20211124,put,33.0,20211227,20211231,37.0,4.0,4
425,20200624,put,12.0,20200706,20200710,16.0,4.0,4
1808,20260520,put,33.0,20260622,20260626,37.0,4.0,4
1809,20260520,call,33.0,20260622,20260626,37.0,4.0,4
9,20180829,call,12.0,20180910,20180907,9.0,-3.0,3
1826,20260605,put,30.0,20260705,20260702,27.0,-3.0,3
8,20180829,put,33.0,20181001,20180928,30.0,-3.0,3
18,20180906,put,33.0,20181009,20181012,36.0,3.0,3
1789,20260429,put,12.0,20260511,20260508,9.0,-3.0,3
1790,20260429,call,12.0,20260511,20260508,9.0,-3.0,3


In [23]:
# ============================================================
# Final QA: strike moneyness
# ============================================================

trade_candidate_final_df["short_strike_moneyness"] = (
    trade_candidate_final_df["short_strike_selected"]
    / trade_candidate_final_df["spx_close"]
)

trade_candidate_final_df["short_strike_pct_otm"] = np.where(
    trade_candidate_final_df["trade_side"] == "put",
    1.0 - trade_candidate_final_df["short_strike_moneyness"],
    trade_candidate_final_df["short_strike_moneyness"] - 1.0,
)

print("Short strike moneyness by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["short_strike_moneyness"]
    .describe()
)

print("\nShort strike pct OTM by side:")
display(
    trade_candidate_final_df
    .groupby("trade_side")["short_strike_pct_otm"]
    .describe()
)

print("\nPut strike examples:")
display(
    trade_candidate_final_df[
        trade_candidate_final_df["trade_side"] == "put"
    ]
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "short_strike_raw",
            "short_strike_selected",
            "short_strike_moneyness",
            "short_strike_pct_otm",
        ]
    ]
    .tail(20)
)

print("\nCall strike examples:")
display(
    trade_candidate_final_df[
        trade_candidate_final_df["trade_side"] == "call"
    ]
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "short_strike_raw",
            "short_strike_selected",
            "short_strike_moneyness",
            "short_strike_pct_otm",
        ]
    ]
    .tail(20)
)

Short strike moneyness by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,1.077405,0.027200,1.030011,1.057866,1.072904,1.092894,1.18487
put,1080.0,0.999391,0.000407,0.997737,0.999137,0.999436,0.999711,1.00000



Short strike pct OTM by side:


,count,mean,std,min,25%,50%,75%,max
trade_side,,,,,,,,
call,749.0,0.077405,0.027200,0.030011,0.057866,0.072904,0.092894,0.184870
put,1080.0,0.000609,0.000407,0.000000,0.000289,0.000564,0.000863,0.002263



Put strike examples:


,trade_date,selected_tenor,actual_dte,spx_close,short_strike_raw,short_strike_selected,short_strike_moneyness,short_strike_pct_otm
1791,20260504,12.0,11.0,7200.75,7200.0,7200.0,0.999896,0.000104
1793,20260505,18.0,17.0,7259.22,7259.0,7255.0,0.999419,0.000581
1795,20260511,24.0,25.0,7412.84,7412.0,7410.0,0.999617,0.000383
1797,20260512,24.0,24.0,7400.96,7400.0,7400.0,0.999870,0.000130
1799,20260513,21.0,23.0,7444.25,7444.0,7440.0,0.999429,0.000571
1802,20260515,30.0,28.0,7408.50,7408.0,7405.0,0.999528,0.000472
1804,20260518,30.0,31.0,7403.05,7403.0,7400.0,0.999588,0.000412
1806,20260519,27.0,30.0,7353.61,7353.0,7350.0,0.999509,0.000491
1808,20260520,33.0,37.0,7432.97,7432.0,7430.0,0.999600,0.000400
1810,20260526,18.0,17.0,7519.12,7519.0,7515.0,0.999452,0.000548



Call strike examples:


,trade_date,selected_tenor,actual_dte,spx_close,signal_implied_vol,short_strike_raw,short_strike_selected,short_strike_moneyness,short_strike_pct_otm
1788,20260428,9.0,10.0,7138.80,16.591230,7530.891433,7550.0,1.057601,0.057601
1790,20260429,12.0,9.0,7135.95,17.422249,7526.395875,7530.0,1.055220,0.055220
1792,20260504,12.0,11.0,7200.75,16.720195,7618.771850,7620.0,1.058223,0.058223
1794,20260505,18.0,17.0,7259.22,16.475916,7775.455141,7800.0,1.074496,0.074496
1796,20260511,24.0,25.0,7412.84,17.804969,8103.683072,8200.0,1.106189,0.106189
1798,20260512,24.0,24.0,7400.96,17.317442,8058.255984,8075.0,1.091075,0.091075
1800,20260513,21.0,23.0,7444.25,16.815346,8072.705080,8075.0,1.084730,0.084730
1801,20260514,27.0,29.0,7501.24,17.010834,8220.591631,8300.0,1.106484,0.106484
1803,20260515,30.0,28.0,7408.50,18.240269,8157.055667,8200.0,1.106837,0.106837
1805,20260518,30.0,31.0,7403.05,17.715762,8167.475104,8170.0,1.103599,0.103599


In [24]:
# ============================================================
# Final QA: call 2SD strike reasonableness
# ============================================================

call_candidate_df = trade_candidate_final_df[
    trade_candidate_final_df["trade_side"] == "call"
].copy()

call_candidate_df["implied_vol_decimal"] = (
    call_candidate_df["signal_implied_vol"] / 100.0
)

call_candidate_df["expected_2sd_pct_move"] = (
    2.0
    * call_candidate_df["implied_vol_decimal"]
    * np.sqrt(call_candidate_df["actual_dte"] / 365.0)
)

call_candidate_df["selected_call_pct_otm"] = (
    call_candidate_df["short_strike_selected"] / call_candidate_df["spx_close"]
    - 1.0
)

call_candidate_df["selected_minus_raw_strike"] = (
    call_candidate_df["short_strike_selected"]
    - call_candidate_df["short_strike_raw"]
)

print("Expected 2SD pct move:")
display(call_candidate_df["expected_2sd_pct_move"].describe())

print("\nSelected call pct OTM:")
display(call_candidate_df["selected_call_pct_otm"].describe())

print("\nSelected strike minus raw strike:")
display(call_candidate_df["selected_minus_raw_strike"].describe())

print("\nLargest selected strike rounding gaps:")
display(
    call_candidate_df
    .sort_values("selected_minus_raw_strike", ascending=False)
    [
        [
            "trade_date",
            "selected_tenor",
            "actual_dte",
            "spx_close",
            "signal_implied_vol",
            "expected_2sd_pct_move",
            "short_strike_raw",
            "short_strike_selected",
            "selected_minus_raw_strike",
            "selected_call_pct_otm",
        ]
    ]
    .head(30)
)

Expected 2SD pct move:


count    749.000000
mean       0.074391
std        0.024572
min        0.027466
25%        0.056584
50%        0.070792
75%        0.089188
max        0.158675
Name: expected_2sd_pct_move, dtype: float64


Selected call pct OTM:


count    749.000000
mean       0.077405
std        0.027200
min        0.030011
25%        0.057866
50%        0.072904
75%        0.092894
max        0.184870
Name: selected_call_pct_otm, dtype: float64


Selected strike minus raw strike:


count    749.000000
mean      13.348689
std       20.943909
min        0.009738
25%        2.573724
50%        5.010610
75%       16.721839
max      189.564973
Name: selected_minus_raw_strike, dtype: float64


Largest selected strike rounding gaps:


,trade_date,selected_tenor,actual_dte,spx_close,signal_implied_vol,expected_2sd_pct_move,short_strike_raw,short_strike_selected,selected_minus_raw_strike,selected_call_pct_otm
965,20230202,27.0,29.0,4179.76,18.277457,0.103038,4610.435027,4800.0,189.564973,0.148391
968,20230207,30.0,31.0,4164.00,18.540327,0.108064,4613.979284,4800.0,186.020716,0.152738
1171,20231213,27.0,30.0,4707.09,12.024246,0.068945,5031.619786,5200.0,168.380214,0.104717
1393,20241010,27.0,29.0,5780.05,20.272835,0.114287,6440.634898,6600.0,159.365102,0.141859
1395,20241011,30.0,28.0,5815.03,20.515309,0.113642,6475.864353,6600.0,124.135647,0.134990
1391,20241009,30.0,30.0,5792.04,20.880061,0.119723,6485.477782,6600.0,114.522218,0.139495
399,20200518,24.0,25.0,2953.91,28.869870,0.151112,3400.280678,3500.0,99.719322,0.184870
1605,20250812,30.0,31.0,6445.76,14.800519,0.086266,7001.812116,7100.0,98.187884,0.101499
1551,20250701,30.0,31.0,6198.01,16.715128,0.097426,6801.856290,6900.0,98.143710,0.113261
1796,20260511,24.0,25.0,7412.84,17.804969,0.093195,8103.683072,8200.0,96.316928,0.106189


In [25]:
# ============================================================
# Final save with QA fields
# ============================================================

TRADE_CANDIDATE_PANEL_CSV = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.csv"
TRADE_CANDIDATE_PANEL_PARQUET = PROCESSED_DATA_DIR / "trade_candidate_panel_v0_1.parquet"

TRADE_CANDIDATE_QA_CSV = AUDIT_DIR / "trade_candidate_qa_v0_1.csv"

trade_candidate_final_df.to_csv(TRADE_CANDIDATE_PANEL_CSV, index=False)
trade_candidate_final_df.to_parquet(TRADE_CANDIDATE_PANEL_PARQUET, index=False)

trade_candidate_qa_df = pd.DataFrame({
    "metric": [
        "rows",
        "mapped_rows",
        "put_rows",
        "call_rows",
        "min_actual_dte",
        "max_actual_dte",
        "mean_expiry_distance_days",
        "max_expiry_distance_days",
        "mean_put_pct_otm",
        "mean_call_pct_otm",
    ],
    "value": [
        len(trade_candidate_final_df),
        (trade_candidate_final_df["mapping_status"] == "mapped").sum(),
        (trade_candidate_final_df["trade_side"] == "put").sum(),
        (trade_candidate_final_df["trade_side"] == "call").sum(),
        trade_candidate_final_df["actual_dte"].min(),
        trade_candidate_final_df["actual_dte"].max(),
        trade_candidate_final_df["expiry_distance_days"].mean(),
        trade_candidate_final_df["expiry_distance_days"].max(),
        trade_candidate_final_df.loc[
            trade_candidate_final_df["trade_side"] == "put",
            "short_strike_pct_otm",
        ].mean(),
        trade_candidate_final_df.loc[
            trade_candidate_final_df["trade_side"] == "call",
            "short_strike_pct_otm",
        ].mean(),
    ],
})

trade_candidate_qa_df.to_csv(TRADE_CANDIDATE_QA_CSV, index=False)

print("Saved final trade candidate CSV:", TRADE_CANDIDATE_PANEL_CSV)
print("Saved final trade candidate parquet:", TRADE_CANDIDATE_PANEL_PARQUET)
print("Saved candidate QA:", TRADE_CANDIDATE_QA_CSV)

display(trade_candidate_qa_df)

Saved final trade candidate CSV: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1.csv
Saved final trade candidate parquet: C:\Users\patri\vrp_project\data\processed\trade_candidate_panel_v0_1.parquet
Saved candidate QA: C:\Users\patri\vrp_project\data\audit\trade_candidate_qa_v0_1.csv


,metric,value
0,rows,1829.000000
1,mapped_rows,1829.000000
2,put_rows,1080.000000
3,call_rows,749.000000
4,min_actual_dte,6.000000
5,max_actual_dte,37.000000
6,mean_expiry_distance_days,1.610169
7,max_expiry_distance_days,4.000000
8,mean_put_pct_otm,0.000609
9,mean_call_pct_otm,0.077405
